# Conceptual Dictionary — Examples

This notebook demonstrates how to use the `conceptual_dictionary` templates to describe two common scenarios:

1. **Single crystal structure with a simulation workflow** — an Al FCC perfect crystal relaxed via molecular statics in LAMMPS.
2. **Grain boundary (GB) structure only** — an Al Σ3 ⟨110⟩ tilt grain boundary with no associated workflow.

Each section builds the in-memory Python dict from the templates, then exports it to **YAML** and **JSON** so the files can be used directly as input to atomRDF's `WorkflowParser`.

In [1]:
import copy, json, yaml, pathlib

from conceptual_dictionary import (
    sample_template,
    workflow_template,
    dataset_template,
    property_template,
    grain_boundary_template,
)

EXAMPLES_DIR = pathlib.Path(".")


## 1 · Single crystal structure with a molecular-statics workflow

We describe two samples (initial and relaxed Al FCC unit cell) and one workflow step that relaxes the structure and records the equilibrium energy and volume.

In [2]:
# ── Initial (unrelaxed) sample ────────────────────────────────────────────────
al_initial = copy.deepcopy(sample_template)
al_initial["id"] = "Al_fcc_initial"
al_initial["material"]["element_ratio"] = {"Al": 1.0}
al_initial["material"]["crystal_structure"].update({
    "spacegroup_symbol": "Fm-3m",
    "spacegroup_number": 225,
})
al_initial["material"]["crystal_structure"]["unit_cell"].update({
    "bravais_lattice": "https://www.wikidata.org/wiki/Q581614",
    "lattice_parameter": [4.05, 4.05, 4.05],
    "angle": [90.0, 90.0, 90.0],
})
al_initial["simulation_cell"].update({
    "number_of_atoms": 4,
    "length": [4.05, 4.05, 4.05],
    "angle": [90.0, 90.0, 90.0],
})
al_initial["atom_attribute"]["species"] = ["Al", "Al", "Al", "Al"]
al_initial["atom_attribute"]["position"] = [
    [0.000, 0.000, 0.000],
    [2.025, 2.025, 0.000],
    [2.025, 0.000, 2.025],
    [0.000, 2.025, 2.025],
]
for k in ("file_path", "file_format", "file_species"):
    al_initial["atom_attribute"].pop(k, None)

# ── Relaxed sample ────────────────────────────────────────────────────────────
lp = 4.032
al_relaxed = copy.deepcopy(al_initial)
al_relaxed["id"] = "Al_fcc_relaxed"
al_relaxed["material"]["crystal_structure"]["unit_cell"]["lattice_parameter"] = [lp, lp, lp]
al_relaxed["simulation_cell"].update({
    "volume": {"value": round(lp**3, 4)},
    "number_of_atoms": 4,
    "length": [lp, lp, lp],
})
half = lp / 2
al_relaxed["atom_attribute"]["position"] = [
    [0.0,  0.0,  0.0],
    [half, half, 0.0],
    [half, 0.0,  half],
    [0.0,  half, half],
]

print("Initial sample id :", al_initial["id"])
print("Relaxed sample id :", al_relaxed["id"])
print("Relaxed lattice parameter:", al_relaxed["material"]["crystal_structure"]["unit_cell"]["lattice_parameter"])


Initial sample id : Al_fcc_initial
Relaxed sample id : Al_fcc_relaxed
Relaxed lattice parameter: [4.032, 4.032, 4.032]


In [3]:
# ── Workflow step ─────────────────────────────────────────────────────────────
ms_workflow = copy.deepcopy(workflow_template)
ms_workflow["method"] = "MolecularStatics"
ms_workflow["degrees_of_freedom"] = ["AtomicPositionRelaxation", "CellVolumeRelaxation"]
ms_workflow["interatomic_potential"] = {
    "potential_type": "eam/alloy",
    "uri": "https://doi.org/10.1103/physrevb.59.3393",
}
ms_workflow["software"] = [{"uri": "https://doi.org/10.1016/j.cpc.2021.108171", "label": "LAMMPS"}]
ms_workflow["input_sample"] = ["Al_fcc_initial"]
ms_workflow["output_sample"] = ["Al_fcc_relaxed"]

e_prop = copy.deepcopy(property_template)
e_prop.update({"label": "EquilibriumEnergy", "value": -3.360, "unit": "EV",
               "associate_to_sample": ["Al_fcc_relaxed"]})

v_prop = copy.deepcopy(property_template)
v_prop.update({"label": "EquilibriumVolume", "value": round(lp**3, 4), "unit": "ANGSTROM3",
               "associate_to_sample": ["Al_fcc_relaxed"]})

ms_workflow["calculated_property"] = [e_prop, v_prop]

# ── Dataset provenance ────────────────────────────────────────────────────────
ds = copy.deepcopy(dataset_template)
ds["title"] = "Al FCC energy-volume relaxation"
ds["creators"] = [{"id": "https://orcid.org/0000-0000-0000-0000", "name": "Jane Doe"}]
ds["samples"] = ["Al_fcc_relaxed"]

# ── Full document ─────────────────────────────────────────────────────────────
single_structure_doc = {
    "dataset": ds,
    "computational_sample": [al_initial, al_relaxed],
    "workflow": [ms_workflow],
}

print("Workflow method      :", single_structure_doc["workflow"][0]["method"])
print("Calculated properties:", [p["label"] for p in single_structure_doc["workflow"][0]["calculated_property"]])


Workflow method      : MolecularStatics
Calculated properties: ['EquilibriumEnergy', 'EquilibriumVolume']


In [4]:
# ── Export: YAML ──────────────────────────────────────────────────────────────
yaml_path = EXAMPLES_DIR / "single_structure_with_workflow.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(single_structure_doc, f, allow_unicode=True, sort_keys=False)

print("Written:", yaml_path)
print("\n--- preview (first 35 lines) ---")
print("\n".join(yaml_path.read_text().splitlines()[:35]))


Written: single_structure_with_workflow.yaml

--- preview (first 35 lines) ---
dataset:
  identifier: null
  title: Al FCC energy-volume relaxation
  creators:
  - id: https://orcid.org/0000-0000-0000-0000
    name: Jane Doe
  publication:
    id: null
    identifier: null
    title: null
  samples:
  - Al_fcc_relaxed
computational_sample:
- id: Al_fcc_initial
  material:
    element_ratio:
      Al: 1.0
    crystal_structure:
      spacegroup_symbol: Fm-3m
      spacegroup_number: 225
      unit_cell:
        bravais_lattice: https://www.wikidata.org/wiki/Q581614
        lattice_parameter:
        - 4.05
        - 4.05
        - 4.05
        angle:
        - 90.0
        - 90.0
        - 90.0
  simulation_cell:
    volume:
      value: null
    number_of_atoms: 4
    length:


In [5]:
# ── Export: JSON ──────────────────────────────────────────────────────────────
json_path = EXAMPLES_DIR / "single_structure_with_workflow.json"
json_path.write_text(json.dumps(single_structure_doc, indent=2))

print("Written:", json_path)
print("\n--- preview (first 35 lines) ---")
print("\n".join(json_path.read_text().splitlines()[:35]))


Written: single_structure_with_workflow.json

--- preview (first 35 lines) ---
{
  "dataset": {
    "identifier": null,
    "title": "Al FCC energy-volume relaxation",
    "creators": [
      {
        "id": "https://orcid.org/0000-0000-0000-0000",
        "name": "Jane Doe"
      }
    ],
    "publication": {
      "id": null,
      "identifier": null,
      "title": null
    },
    "samples": [
      "Al_fcc_relaxed"
    ]
  },
  "computational_sample": [
    {
      "id": "Al_fcc_initial",
      "material": {
        "element_ratio": {
          "Al": 1.0
        },
        "crystal_structure": {
          "spacegroup_symbol": "Fm-3m",
          "spacegroup_number": 225,
          "unit_cell": {
            "bravais_lattice": "https://www.wikidata.org/wiki/Q581614",
            "lattice_parameter": [
              4.05,
              4.05,
              4.05


## 2 · Grain boundary structure only

No workflow here — just two GB samples (two variants of an Al Σ3 ⟨110⟩ tilt boundary). The key difference from the single-crystal case is the `tilt_grain_boundary` block inside each sample, built from `grain_boundary_template`.

In [6]:
def make_gb_sample(sample_id, lp_xyz, n_atoms, file_path, sigma, plane, misorientation_angle, rotation_axis):
    """Return a sample dict with a tilt_grain_boundary block."""
    s = copy.deepcopy(sample_template)
    s["id"] = sample_id
    s["material"]["element_ratio"] = {"Al": 1.0}
    s["material"]["crystal_structure"].update({
        "spacegroup_symbol": "Fm-3m",
        "spacegroup_number": 225,
    })
    s["material"]["crystal_structure"]["unit_cell"].update({
        "bravais_lattice": "https://www.wikidata.org/wiki/Q581614",
        "lattice_parameter": lp_xyz,
        "angle": [90.0, 90.0, 90.0],
    })
    s["simulation_cell"].update({
        "number_of_atoms": n_atoms,
        "length": lp_xyz,
        "angle": [90.0, 90.0, 90.0],
    })
    s["atom_attribute"] = {
        "file_path": file_path,
        "file_format": "lammps-data",
        "file_species": ["Al"],
    }
    gb = copy.deepcopy(grain_boundary_template)
    gb.update({
        "sigma": sigma,
        "plane": plane,
        "misorientation_angle": misorientation_angle,
        "rotation_axis": rotation_axis,
    })
    s["tilt_grain_boundary"] = gb
    return s


gb1 = make_gb_sample(
    sample_id="Al_sigma3_gb1",
    lp_xyz=[7.015, 117.967, 5.728],
    n_atoms=284,
    file_path="./data/Al_sigma3_gb1.lammps",
    sigma=3,
    plane=[1.0, 1.0, 2.0],
    misorientation_angle=109.47,
    rotation_axis=[1, 1, 0],
)

gb2 = make_gb_sample(
    sample_id="Al_sigma3_gb2",
    lp_xyz=[98.707, 136.841, 5.728],
    n_atoms=4640,
    file_path="./data/Al_sigma3_gb2.lammps",
    sigma=3,
    plane=[2.0, 2.0, 3.0],
    misorientation_angle=38.94,
    rotation_axis=[1, 1, 0],
)

print("GB1 id:", gb1["id"], "| sigma:", gb1["tilt_grain_boundary"]["sigma"],
      "| plane:", gb1["tilt_grain_boundary"]["plane"])
print("GB2 id:", gb2["id"], "| sigma:", gb2["tilt_grain_boundary"]["sigma"],
      "| plane:", gb2["tilt_grain_boundary"]["plane"])


GB1 id: Al_sigma3_gb1 | sigma: 3 | plane: [1.0, 1.0, 2.0]
GB2 id: Al_sigma3_gb2 | sigma: 3 | plane: [2.0, 2.0, 3.0]


In [7]:
gb_doc = {"computational_sample": [gb1, gb2]}

# ── Export: YAML ──────────────────────────────────────────────────────────────
gb_yaml_path = EXAMPLES_DIR / "grain_boundary.yaml"
with open(gb_yaml_path, "w") as f:
    yaml.dump(gb_doc, f, allow_unicode=True, sort_keys=False)

print("Written:", gb_yaml_path)
print("\n--- tilt_grain_boundary block (gb1) ---")
loaded = yaml.safe_load(gb_yaml_path.read_text())["computational_sample"][0]
print(yaml.dump({"tilt_grain_boundary": loaded["tilt_grain_boundary"]}, sort_keys=False))


Written: grain_boundary.yaml

--- tilt_grain_boundary block (gb1) ---
tilt_grain_boundary:
  sigma: 3
  plane:
  - 1.0
  - 1.0
  - 2.0
  misorientation_angle: 109.47
  rotation_axis:
  - 1
  - 1
  - 0



In [8]:
# ── Export: JSON ──────────────────────────────────────────────────────────────
gb_json_path = EXAMPLES_DIR / "grain_boundary.json"
gb_json_path.write_text(json.dumps(gb_doc, indent=2))

print("Written:", gb_json_path)
print("\n--- tilt_grain_boundary block (gb1, JSON) ---")
gb1_json = json.loads(gb_json_path.read_text())["computational_sample"][0]
print(json.dumps(gb1_json["tilt_grain_boundary"], indent=2))


Written: grain_boundary.json

--- tilt_grain_boundary block (gb1, JSON) ---
{
  "sigma": 3,
  "plane": [
    1.0,
    1.0,
    2.0
  ],
  "misorientation_angle": 109.47,
  "rotation_axis": [
    1,
    1,
    0
  ]
}
